In [1]:
# ----- Setup -----
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
# Add at the top with imports
import random



# ----- Load Zara 2 Data -----
df = pd.read_csv("converted_zara_2.csv")
x_mean, x_std = df['x'].mean(), df['x'].std()
y_mean, y_std = df['y'].mean(), df['y'].std()
df['x'] = (df['x'] - x_mean) / x_std
df['y'] = (df['y'] - y_mean) / y_std

SEQ_LEN = 8
PRED_LEN = 12

# ----- Dataset Definition -----
class TrajectoryDataset(Dataset):
    def __init__(self, trajectories):
        self.trajectories = trajectories

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        obs, fut = self.trajectories[idx]
        return torch.tensor(obs, dtype=torch.float32), torch.tensor(fut, dtype=torch.float32)


# ----- Rebuild Zara 2 Trajectories -----
trajectories = []
for pid, person_df in df.groupby("person_id"):
    person_df = person_df.sort_values("frame_id")
    coords = person_df[['x', 'y']].values
    for i in range(len(coords) - SEQ_LEN - PRED_LEN):
        obs = coords[i:i+SEQ_LEN]
        fut = coords[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN]
        trajectories.append((obs, fut))

data_loader = DataLoader(TrajectoryDataset(trajectories), batch_size=64, shuffle=False)

# ----- Load Metadata from Training -----
with open("results_metadata.json", "r") as f:  #----!
    metadata = pd.read_json(f)  #----!

# ----- LSTM Model Definition (same as before) -----
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_len, num_layers, dropout, bidirectional):
        super(LSTMModel, self).__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True, bidirectional=bidirectional
        )
        direction_multiplier = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_size * direction_multiplier, 2 * output_len)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        hn = torch.cat((hn[-2], hn[-1]), dim=1) if self.bidirectional else hn[-1]
        out = self.fc(hn)
        return out.view(-1, self.fc.out_features // 2, 2)

        # ----- Evaluate Saved Models on Zara 2 -----
results = []
criterion = nn.MSELoss()

for _, row in metadata.iterrows():
    exp_id = 37 # change this number to the model you want to see
    #exp_id = row["experiment_id"]
    row = metadata[metadata["experiment_id"] == exp_id].iloc[0]
    #model_path = f"lstm_zara_exp_{exp_id}.pth"
    if not os.path.exists(model_path):
        continue

    model = LSTMModel(
    input_size=2,
    hidden_size=int(row["hidden_size"]),
    output_len=int(row["pred_len"]),
    num_layers=int(row["num_layers"]),
    dropout=float(row["dropout"]),
    bidirectional=bool(row["bidirectional"])
    )
    model.load_state_dict(torch.load(f"lstm_zara_exp_{exp_id}.pth"))
    #model.load_state_dict(torch.load(model_path))
    model.eval()

    total_loss = 0
    with torch.no_grad():
        for obs, fut in data_loader:
            pred = model(obs)
            loss = criterion(pred, fut)
            total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    results.append({**row.to_dict(), "zara2_loss": avg_loss})  #----!

IndexError: single positional indexer is out-of-bounds

In [ ]:
# ----- Real-Time Predicted vs Actual Visualization -----
def visualize_predictions(model, data_loader, num_samples=5):
    model.eval()
    samples_shown = 0
    with torch.no_grad():
        for obs, fut in data_loader:
            pred = model(obs)
            for i in range(min(num_samples, obs.shape[0])):
                obs_np = obs[i].cpu().numpy()
                fut_np = fut[i].cpu().numpy()
                pred_np = pred[i].cpu().numpy()

                full_actual = np.vstack((obs_np, fut_np))
                full_pred = np.vstack((obs_np, pred_np))

                plt.figure(figsize=(6, 6))
                plt.plot(full_actual[:, 0], full_actual[:, 1], 'o-', label='Actual')
                plt.plot(full_pred[:, 0], full_pred[:, 1], 'x--', label='Predicted')
                plt.scatter(obs_np[-1, 0], obs_np[-1, 1], color='black', label='Last Obs', zorder=5)
                plt.title(f"Prediction vs Actual (Sample {samples_shown+1})")
                plt.legend()
                plt.xlabel("x")
                plt.ylabel("y")
                plt.grid(True)
                plt.axis("equal")
                plt.show()

                samples_shown += 1
                if samples_shown >= num_samples:
                    return

In [ ]:
# ----- Call Visualization for Best Model -----
best_model_row = metadata.sort_values("test_loss").iloc[0]  # or sort by "zara2_loss" if already evaluated
best_model = LSTMModel(
    input_size=2,
    hidden_size=int(best_model_row["hidden_size"]),
    output_len=int(best_model_row["pred_len"]),
    num_layers=int(best_model_row["num_layers"]),
    dropout=float(best_model_row["dropout"]),
    bidirectional=bool(best_model_row["bidirectional"])
)
best_model.load_state_dict(torch.load(f"lstm_zara_exp_{best_model_row['experiment_id']}.pth"))
visualize_predictions(best_model, data_loader, num_samples=5)
